In [22]:
import pandas as pd
import ast

Gathering Data

In [23]:
df = pd.read_csv('SynCV-JobPred.csv')
display(df.head(3))

,age,years_experience,education_level,major,job,summary,experience_desc
0,27.0,8.3,master,NaN,teacher,I am a master graduate in arts with 7 years of...,Worked on projects: mentorship program. Achiev...
1,NaN,4.4,master,arts,teacher,I am a master graduate in arts with 4 years of...,Involved in mentorship program; teaching curri...
2,30.0,8.8,bachelor,business,data_analyst,Holding a bachelor in business and bringing 8 ...,Involved in dashboard project; sales predictio...


Assesing Data

In [24]:
df.info()
print("\nMissing values tiap kolom:\n", df.isna().sum())
print("\nJumlah duplikasi: ", df.duplicated().sum())
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318362 entries, 0 to 318361
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   age               273242 non-null  float64
 1   years_experience  272883 non-null  float64
 2   education_level   273210 non-null  object 
 3   major             272835 non-null  object 
 4   job               272778 non-null  object 
 5   summary           273287 non-null  object 
 6   experience_desc   272669 non-null  object 
dtypes: float64(2), object(5)
memory usage: 17.0+ MB

Missing values tiap kolom:
 age                 45120
years_experience    45479
education_level     45152
major               45527
job                 45584
summary             45075
experience_desc     45693
dtype: int64

Jumlah duplikasi:  17239


,age,years_experience
count,273242.000000,272883.000000
mean,27.303647,5.719090
std,6.418563,4.955672
min,8.000000,0.000000
25%,24.000000,1.000000
50%,27.000000,5.000000
75%,30.000000,9.100000
max,208.000000,32.000000


Cleaning Data

In [25]:
# Drop duplikat dan target yang kosong
df = df.drop_duplicates()
df = df.dropna(subset=['job'])

# Batasi rentang angka
if 'age' in df.columns:
    df['age'] = df['age'].clip(lower=18, upper=60)
if 'skill_score' in df.columns:
    df['skill_score'] = df['skill_score'].clip(lower=0, upper=100)

print("Shape sementara:", df.shape)

Shape sementara: (257058, 7)


In [26]:
# Handle kolom tipe list
def parse_list(x):
    if pd.isna(x): return []
    try: return ast.literal_eval(x)
    except: return []

for col in ['skills', 'certifications', 'projects']:
    if col in df.columns:
        df[col] = df[col].apply(parse_list)

# Isi missing value numerik (Median)
for col in ['age', 'years_experience', 'skill_score']:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [27]:
# Isi missing value teks
for col in ['education_level', 'major', 'summary', 'experience_desc']:
    if col in df.columns:
        df[col] = df[col].fillna('unknown')

# Case folding (Huruf kecil)
for col in ['job', 'education_level', 'major']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()

# Hapus duplikat akhir
df = df[~df.astype(str).duplicated()]
print("Shape final:", df.shape)

Shape final: (256246, 7)


Export Data

In [28]:
print("1. Cek Info dan Tipe Data")
df.info()

print("\n2. Cek Sisa Data Kosong dan Duplikat")
print("Sisa data kosong:", df.isna().sum().sum())
print("Sisa data duplikat:", df.astype(str).duplicated().sum())

print("\n3. Cek Rentang Nilai Numerik")
if 'age' in df.columns:
    print("Umur terkecil dan terbesar:", df['age'].min(), "-", df['age'].max())
if 'skill_score' in df.columns:
    print("Skor terkecil dan terbesar:", df['skill_score'].min(), "-", df['skill_score'].max())

print("\n4. Lihat 3 Baris Teratas")
display(df.head(3))

df.to_csv('Data_After_Wrangling.csv', index=False)

1. Cek Info dan Tipe Data
<class 'pandas.core.frame.DataFrame'>
Index: 256246 entries, 0 to 318321
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   age               256246 non-null  float64
 1   years_experience  256246 non-null  float64
 2   education_level   256246 non-null  object 
 3   major             256246 non-null  object 
 4   job               256246 non-null  object 
 5   summary           256246 non-null  object 
 6   experience_desc   256246 non-null  object 
dtypes: float64(2), object(5)
memory usage: 15.6+ MB

2. Cek Sisa Data Kosong dan Duplikat
Sisa data kosong: 0
Sisa data duplikat: 0

3. Cek Rentang Nilai Numerik
Umur terkecil dan terbesar: 18.0 - 60.0

4. Lihat 3 Baris Teratas


,age,years_experience,education_level,major,job,summary,experience_desc
0,27.0,8.3,master,unknown,teacher,I am a master graduate in arts with 7 years of...,Worked on projects: mentorship program. Achiev...
1,27.0,4.4,master,arts,teacher,I am a master graduate in arts with 4 years of...,Involved in mentorship program; teaching curri...
2,30.0,8.8,bachelor,business,data_analyst,Holding a bachelor in business and bringing 8 ...,Involved in dashboard project; sales predictio...
